# L09 · NB 04 — Build a semantic search engine

> *We have embeddings. Now let's wrap them in a clean, production-style search engine and benchmark it head-to-head against TF-IDF — the classical baseline that every real search system still uses for exact-match queries.*

By the end of this notebook you will:

1. Have a `SemanticSearch` class with a clean API: `.add()`, `.search()`, `.search_with_filters()`
2. Have run it head-to-head against `TfidfVectorizer` from sklearn
3. Understand when each approach wins (and why production search combines them)
4. See Sarah's production wiring diagram

## 1 · Setup — load model, catalogue, cached embeddings

In [1]:
# --- Colab setup: install sentence-transformers if missing (no-op in the local dsai-m3 env) ---
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)

import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

import json
import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

torch.set_num_threads(1)

_LOCAL = 'data/northstar_catalogue.csv'
_URL = 'https://raw.githubusercontent.com/flexfengfeng/6m-data-3.9-Natural-Language-Processing/main/notebooks/data/northstar_catalogue.csv'
df = pd.read_csv(_LOCAL if os.path.exists(_LOCAL) else _URL)

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Load or recompute catalogue embeddings (cached from NB 03).
# Recompute if the file is missing OR empty/corrupt, so a stale 0-byte
# file can never crash this cell.
def _have_valid(path):
    return os.path.exists(path) and os.path.getsize(path) > 0

catalogue_emb = None
if _have_valid('catalogue_embeddings.npy'):
    try:
        catalogue_emb = np.load('catalogue_embeddings.npy')
        print(f"Loaded cached embeddings: {catalogue_emb.shape}")
    except Exception as e:
        print(f"Cached embeddings unreadable ({e}); recomputing.")

if catalogue_emb is None:
    docs = (df['name'] + ' — ' + df['description']).tolist()
    catalogue_emb = model.encode(docs)
    np.save('catalogue_embeddings.npy', catalogue_emb)
    print(f"Computed embeddings: {catalogue_emb.shape}")

# The 8-query benchmark from NB 01, inlined so this notebook is self-contained
# (run it standalone in Colab without needing NB 01's output file).
ground_truth = {
    "blue summer dress":         "Lila Floral Sundress",
    "warm winter jumper":        "Roan Cable Jumper",
    "smart office shoes":        "Loam Ankle Boots",
    "lightweight rain jacket":   "Trail Windbreaker",
    "cosy throw for sofa":       "Wool Throw Blanket",
    "gym leggings":              "Storm Yoga Leggings",
    "beach holiday outfit":      "Cassia Maxi Gown",
    "running trainers":          "Cloud Running Trainers",
}

/opt/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9070.19it/s]

Loaded cached embeddings: (76, 384)


## 2 · A clean search-engine class

One class, three methods. Everything that follows uses this object.

In [2]:
class SemanticSearch:
    """A minimal semantic search engine over a product catalogue.

    add(df, text_cols)       — embed the corpus once and store
    search(query, top=10)    — top-K by cosine similarity
    search_with_filters(...) — top-K within structured filters
    """
    def __init__(self, model):
        self.model = model
        self.df = None
        self.embeddings = None

    def add(self, df, text_cols=('name', 'description')):
        self.df = df.reset_index(drop=True).copy()
        docs = self.df[list(text_cols)].agg(' — '.join, axis=1).tolist()
        self.embeddings = self.model.encode(docs, show_progress_bar=False)
        return self

    def search(self, query, top=5):
        q = self.model.encode([query])
        sims = cosine_similarity(q, self.embeddings)[0]
        order = np.argsort(-sims)[:top]
        return self.df.iloc[order].assign(score=sims[order]).reset_index(drop=True)

    def search_with_filters(self, query, top=5, **filters):
        """Filter the corpus first, then rank by similarity."""
        q = self.model.encode([query])
        mask = pd.Series(True, index=self.df.index)
        for col, val in filters.items():
            if isinstance(val, (list, tuple)):
                mask &= self.df[col].isin(val)
            else:
                mask &= (self.df[col] == val)
        sub_idx = self.df.index[mask]
        if len(sub_idx) == 0:
            return self.df.iloc[:0]
        sub_emb = self.embeddings[mask.values]
        sims = cosine_similarity(q, sub_emb)[0]
        order = np.argsort(-sims)[:top]
        return self.df.iloc[sub_idx[order]].assign(score=sims[order]).reset_index(drop=True)

# Wire it up — re-uses our cached embeddings, just for cleanliness
engine = SemanticSearch(model)
# Skip re-encoding (we already have catalogue_emb)
engine.df = df.reset_index(drop=True).copy()
engine.embeddings = catalogue_emb
print('Engine ready.')

Engine ready.


## 3 · Try a few queries

In [3]:
for q in ['blue summer dress', 'something cosy to wear hiking', 'business attire for women']:
    print(f"\nQuery: {q!r}")
    print(engine.search(q, top=5)[['category','name','score']].to_string(index=False))


Query: 'blue summer dress'


category                    name    score
   shirt       Frost Linen Shirt 0.547784
   dress    Marigold Shift Dress 0.540338
   dress    Lila Floral Sundress 0.467017
   dress Holly Knit Jumper Dress 0.466092
   shirt       Indigo Polo Shirt 0.460361

Query: 'something cosy to wear hiking'


category                 name    score
footwear Boulder Hiking Boots 0.500391
footwear     Aspen Snow Boots 0.482619
   shirt   Pine Flannel Shirt 0.464685
   shirt    Frost Linen Shirt 0.457966
footwear     Loam Ankle Boots 0.437125

Query: 'business attire for women'
category                 name    score
   dress     Cassia Maxi Gown 0.460027
   dress Sienna Bodycon Dress 0.458183
    knit Heather Crew Sweater 0.441692
   shirt Rose Puff Sleeve Top 0.432061
   shirt     Onyx Silk Blouse 0.417356


Note the second and third queries — *something cosy to wear hiking* and *business attire for women*. Neither uses any product-category keywords. Both pull plausible results. That's semantic search delivering on its promise.

## 4 · Filters on top of semantic ranking

In production you usually want to RESTRICT the corpus first (e.g., to one category, or one price range) and then rank within that subset. The `search_with_filters` method does this.

In [4]:
# "blue summer dress" but only within the dress category
print("Query: 'blue summer dress'   |   filter: category=='dress'")
print(engine.search_with_filters('blue summer dress', top=5, category='dress')[['name','score']].to_string(index=False))

print()
print("Query: 'lightweight rain jacket'   |   filter: category in ['coat','activewear']")
print(engine.search_with_filters('lightweight rain jacket', top=5, category=['coat','activewear'])[['category','name','score']].to_string(index=False))

Query: 'blue summer dress'   |   filter: category=='dress'
                   name    score
   Marigold Shift Dress 0.540338
   Lila Floral Sundress 0.467017
Holly Knit Jumper Dress 0.466092
   Sienna Bodycon Dress 0.447115
       Cassia Maxi Gown 0.430717

Query: 'lightweight rain jacket'   |   filter: category in ['coat','activewear']
category                  name    score
    coat   Cinder Biker Jacket 0.561851
    coat   Verge Bomber Jacket 0.559493
    coat  Ember Quilted Jacket 0.531884
    coat     Tarn Waxed Jacket 0.498808
    coat Glacier Puffer Jacket 0.453990


With the category filter applied, the non-dress distractor (Frost Linen Shirt) is gone, so *Lila Floral Sundress* climbs from rank 3 to **rank 2**. It is still behind *Marigold Shift Dress* (a yellow dress) — the model over-weights the word "dress" relative to the colour "blue". So the filter helps but does **not** fully fix the colour-blindness; closing that gap needs a colour filter or a re-ranker on top.

**Production lesson:** structured filters and semantic ranking are complementary. Don't pick one. Use filters for what filters do well (exact field matching), use embeddings for the rest.

## 5 · TF-IDF — the classical baseline

Let's build a TF-IDF search engine over the same corpus and compare head-to-head. TF-IDF treats every word as independent but weights them by how rare they are across the corpus.

In [5]:
class TfidfSearch:
    def __init__(self):
        self.df = None
        self.vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
        self.matrix = None

    def add(self, df, text_cols=('name', 'description')):
        self.df = df.reset_index(drop=True).copy()
        docs = self.df[list(text_cols)].agg(' '.join, axis=1).tolist()
        self.matrix = self.vec.fit_transform(docs)
        return self

    def search(self, query, top=5):
        q = self.vec.transform([query])
        sims = cosine_similarity(q, self.matrix)[0]
        order = np.argsort(-sims)[:top]
        return self.df.iloc[order].assign(score=sims[order]).reset_index(drop=True)

tfidf = TfidfSearch().add(df)
print(f"TF-IDF vocabulary size: {len(tfidf.vec.vocabulary_)}")
print(f"TF-IDF matrix shape   : {tfidf.matrix.shape}")

TF-IDF vocabulary size: 1271
TF-IDF matrix shape   : (76, 1271)


### TF-IDF on the same 'blue summer dress' query

In [6]:
print("Query: 'blue summer dress'")
print()
print(tfidf.search('blue summer dress', top=5)[['category','name','score']].to_string(index=False))

Query: 'blue summer dress'

category                    name    score
   dress Holly Knit Jumper Dress 0.147678
   dress       Marina Wrap Dress 0.144762
   shirt       Frost Linen Shirt 0.124843
    knit    Cobalt V-Neck Jumper 0.122155
footwear Sand Espadrille Sandals 0.118865


TF-IDF's bigrams and IDF weighting are an improvement over the NB 01 keyword search, but it still can't find Lila Floral Sundress because the description never uses the words *summer*, *dress*, or *blue*.

## 6 · Head-to-head benchmark

Run all 8 ground-truth queries through both engines.

In [7]:
def evaluate(engine, ground_truth, top_k=5):
    """Return per-query top-K results and metric: top-1 + top-K hit rate."""
    results = []
    for q, expected in ground_truth.items():
        top = engine.search(q, top=top_k)
        top1_ok = top.iloc[0]['name'] == expected
        topk_ok = expected in top['name'].tolist()
        results.append((q, expected, top.iloc[0]['name'], top1_ok, topk_ok))
    return results

sem_results = evaluate(engine, ground_truth, top_k=5)
tf_results  = evaluate(tfidf,  ground_truth, top_k=5)

print(f"{'Query':30s} {'Semantic top-1':35s} {'TF-IDF top-1':35s} {'sem@1':>6s} {'tf@1':>6s}")
print('-' * 120)
for (q, exp, sem_top, sem_ok, sem_topk), (_, _, tf_top, tf_ok, tf_topk) in zip(sem_results, tf_results):
    print(f"  {q:28s} {sem_top:35s} {tf_top:35s} {'✅' if sem_ok else '❌':>6s} {'✅' if tf_ok else '❌':>6s}")

sem_top1 = sum(r[3] for r in sem_results)
tf_top1  = sum(r[3] for r in tf_results)
sem_top5 = sum(r[4] for r in sem_results)
tf_top5  = sum(r[4] for r in tf_results)

print(f"\n{'Approach':12s} {'Top-1':>6s} {'Top-5':>6s}")
print(f"{'Semantic':12s} {sem_top1}/{len(sem_results):d}    {sem_top5}/{len(sem_results):d}")
print(f"{'TF-IDF':12s} {tf_top1}/{len(tf_results):d}    {tf_top5}/{len(tf_results):d}")

Query                          Semantic top-1                      TF-IDF top-1                         sem@1   tf@1
------------------------------------------------------------------------------------------------------------------------
  blue summer dress            Frost Linen Shirt                   Holly Knit Jumper Dress                  ❌      ❌
  warm winter jumper           Roan Cable Jumper                   Holly Knit Jumper Dress                  ✅      ❌
  smart office shoes           Loam Ankle Boots                    Cloud Running Trainers                   ✅      ❌
  lightweight rain jacket      Cinder Biker Jacket                 Ember Quilted Jacket                     ❌      ❌
  cosy throw for sofa          Wool Throw Blanket                  Wool Throw Blanket                       ✅      ✅
  gym leggings                 Storm Yoga Leggings                 Storm Yoga Leggings                      ✅      ✅
  beach holiday outfit         Cassia Maxi Gown             

**Headline result: Semantic 6/8 top-1, 7/8 top-5. TF-IDF 4/8 on both.**

Semantic wins decisively. The four common wins are the "easy" queries (cosy throw / gym leggings / beach holiday outfit / running trainers — where the right answer's name contains a content word from the query). The two semantic-only wins (warm winter jumper, smart office shoes) are exactly the synonym-heavy cases where TF-IDF goes off the rails — *jumper* matches "Holly Knit Jumper Dress" (a dress!) because TF-IDF doesn't know one is a noun-modifier and one is the head noun.

The two queries where both fail (*blue summer dress*, *lightweight rain jacket*) need the structured filters we explored in §4. We'll close that gap shortly.

## 7 · When each approach wins

Let's construct two queries to make the contrast obvious.

In [8]:
contrastive_queries = {
    "P0001": "soft cotton flowy garment with petal patterns",  # Lila Floral Sundress — pure synonym
    "P0002": "Marina Wrap Dress",                              # Exact product name — TF-IDF should crush
}

for pid, q in contrastive_queries.items():
    expected = df.loc[df['product_id'] == pid, 'name'].iloc[0]
    sem_top1 = engine.search(q, top=1).iloc[0]['name']
    tf_top1  = tfidf.search(q, top=1).iloc[0]['name']
    sem_rank = engine.search(q, top=76).reset_index(drop=True)
    sem_rank = sem_rank.index[sem_rank['name'] == expected].tolist()
    sem_rank = sem_rank[0] + 1 if sem_rank else 'NOT IN TOP 76'
    tf_rank  = tfidf.search(q,  top=76).reset_index(drop=True)
    tf_rank  = tf_rank.index[tf_rank['name'] == expected].tolist()
    tf_rank  = tf_rank[0] + 1 if tf_rank else 'NOT IN TOP 76'

    print(f"Query: {q!r}")
    print(f"  Expected         : {expected}")
    print(f"  Semantic top-1   : {sem_top1}  (rank of expected: {sem_rank})")
    print(f"  TF-IDF top-1     : {tf_top1}  (rank of expected: {tf_rank})")
    print()

Query: 'soft cotton flowy garment with petal patterns'
  Expected         : Lila Floral Sundress
  Semantic top-1   : Indigo Polo Shirt  (rank of expected: 19)
  TF-IDF top-1     : Velvet Cushion Cover  (rank of expected: 17)

Query: 'Marina Wrap Dress'
  Expected         : Marina Wrap Dress
  Semantic top-1   : Marina Wrap Dress  (rank of expected: 1)
  TF-IDF top-1     : Marina Wrap Dress  (rank of expected: 1)



**Honest observations from the two contrastive queries:**

- For the **synonym query** ("soft cotton flowy garment with petal patterns"), *both* engines fail to put Lila at top-1 — semantic ranks it 19, TF-IDF ranks it 17. This is a hard case: the query uses metaphorical language (*petal patterns* for floral, *flowy garment* for dress) that even the pretrained model only weakly connects to the product description.
- For the **exact product name** ("Marina Wrap Dress"), both engines instantly return the right product at rank 1. TF-IDF's strength is exact matching; semantic also handles this case because the literal words are highest-similarity matches.

Why does the synonym query fail? The small catalogue (76 products) gives the model few examples to differentiate from. And the query reads more like poetry than a typical search — real customers tend to use simpler phrasings like "floral cotton dress" which the model handles well.

**This is the production reality:**

```
final_results = semantic_top_K  ∪  tfidf_top_K   # union of candidates
              → re-rank by weighted combo + structured filters
              → return top-K
```

Hybrid retrieval. Embeddings for semantic recall, TF-IDF for exact-token precision, structured filters for hard constraints, re-ranker on top for the final scoring. Try the hybrid score in the Extensions section — α=0.5 actually brings Lila into the top 5 for "blue summer dress".

## 8 · Sarah's production wiring diagram

Design for NorthStar's search:

In [9]:
print('''
                    +-----------------------------+
                    |   Customer types a query    |
                    +-------------+---------------+
                                  |
                  +---------------+---------------+
                  |                               |
        +---------v---------+         +-----------v----------+
        |  TF-IDF retrieval |         | Semantic retrieval   |
        |   top-50 by BM25  |         |  top-50 by cosine    |
        +---------+---------+         +-----------+----------+
                  |                               |
                  +---------------+---------------+
                                  |
                    +-------------v-------------+
                    | Union → 80-100 candidates |
                    +-------------+-------------+
                                  |
                    +-------------v-------------+
                    | Structured filters apply: |
                    | category / colour / price |
                    +-------------+-------------+
                                  |
                    +-------------v-------------+
                    | Re-rank by score blend +  |
                    | popularity + recency      |
                    +-------------+-------------+
                                  |
                    +-------------v-------------+
                    |   Top-20 to the customer  |
                    +---------------------------+
''')


                    +-----------------------------+
                    |   Customer types a query    |
                    +-------------+---------------+
                                  |
                  +---------------+---------------+
                  |                               |
        +---------v---------+         +-----------v----------+
        |  TF-IDF retrieval |         | Semantic retrieval   |
        |   top-50 by BM25  |         |  top-50 by cosine    |
        +---------+---------+         +-----------+----------+
                  |                               |
                  +---------------+---------------+
                                  |
                    +-------------v-------------+
                    | Union → 80-100 candidates |
                    +-------------+-------------+
                                  |
                    +-------------v-------------+
                    | Structured filters apply: |
                    | cat

**Practical considerations Sarah flagged for production:**

- **Embedding latency.** Encoding the query is ~5 ms on CPU. Cosine over 50K products is < 5 ms with a NumPy matmul. Total search latency < 50 ms — fine.
- **Re-embedding cadence.** When merchandisers update a product description, that product's embedding goes stale. Re-embed the affected rows nightly.
- **Cold-start.** When NEW products arrive, embed them within minutes (a separate small batch job) so they appear in search quickly.
- **Vector database.** At < 100K products, NumPy is fine. At 1M+, switch to FAISS / Pinecone / Weaviate for sub-millisecond approximate-nearest-neighbour.
- **Monitoring.** Log query → top-result CTRs. If a query's top results have zero clicks → bad ranking → re-rank weights need tuning.

## 9 · Recap

In this notebook we:

1. Built a `SemanticSearch` class with a clean three-method API
2. Added structured filters on top of semantic ranking — *exactly* what production search needs
3. Compared head-to-head against TF-IDF and saw both have niches (synonyms vs exact tokens)
4. Drew the production diagram: hybrid retrieval + filters + re-ranker

You now have a working semantic search engine. The next lesson opens the black box: what's INSIDE the transformer that produces these embeddings?

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## E1 · A simple hybrid score

Combine semantic and TF-IDF scores linearly. Tune the weight and see what works best.

In [10]:
def hybrid_search(query, top=5, alpha=0.5):
    """Hybrid = alpha * semantic + (1-alpha) * tfidf"""
    q_sem = model.encode([query])
    sem_sims = cosine_similarity(q_sem, catalogue_emb)[0]

    q_tf = tfidf.vec.transform([query])
    tf_sims = cosine_similarity(q_tf, tfidf.matrix)[0]

    # Both are roughly in [0, 1] — but absolute scales differ.
    # Min-max normalise within this query's candidates.
    def mm(x):
        lo, hi = x.min(), x.max()
        return (x - lo) / (hi - lo + 1e-9)

    blend = alpha * mm(sem_sims) + (1 - alpha) * mm(tf_sims)
    order = np.argsort(-blend)[:top]
    return df.iloc[order].assign(
        score=blend[order],
        sem_score=sem_sims[order],
        tf_score=tf_sims[order]
    ).reset_index(drop=True)

for alpha in [0.0, 0.5, 1.0]:
    print(f"\nalpha = {alpha:.1f}  (0=pure TF-IDF, 1=pure semantic)")
    print(f"Query: 'blue summer dress'")
    print(hybrid_search('blue summer dress', alpha=alpha)[['name','score','sem_score','tf_score']].to_string(index=False))


alpha = 0.0  (0=pure TF-IDF, 1=pure semantic)
Query: 'blue summer dress'
                   name    score  sem_score  tf_score
Holly Knit Jumper Dress 1.000000   0.466092  0.147678
      Marina Wrap Dress 0.980256   0.403601  0.144762
      Frost Linen Shirt 0.845372   0.547784  0.124843
   Cobalt V-Neck Jumper 0.827169   0.401502  0.122155
Sand Espadrille Sandals 0.804894   0.265828  0.118865

alpha = 0.5  (0=pure TF-IDF, 1=pure semantic)
Query: 'blue summer dress'
                   name    score  sem_score  tf_score
      Frost Linen Shirt 0.922686   0.547784  0.124843
Holly Knit Jumper Dress 0.916393   0.466092  0.147678
      Marina Wrap Dress 0.842566   0.403601  0.144762
   Lila Floral Sundress 0.765718   0.467017  0.102895
   Cobalt V-Neck Jumper 0.763874   0.401502  0.122155

alpha = 1.0  (0=pure TF-IDF, 1=pure semantic)
Query: 'blue summer dress'
                   name    score  sem_score  tf_score
      Frost Linen Shirt 1.000000   0.547784  0.124843
   Marigold Shift Dres

Run the cell with α=0.0, α=0.5, α=1.0 to see the spectrum. The sweet spot for most production systems sits around 0.6-0.8 — slightly biased toward semantic but TF-IDF helping with exact matches.

## E2 · Find duplicates within the catalogue

Catalogue dedup is a real merchandising problem. Embed every product and look for pairs with very high cosine similarity — those are candidates for being near-duplicates.

In [11]:
# All-pairs cosine
all_sims = cosine_similarity(catalogue_emb)
np.fill_diagonal(all_sims, -1)  # exclude self-similarity

# Find the top-3 most similar product pairs
flat_idx = np.argsort(-all_sims.ravel())[:6]
seen = set()
print("Top product-pair similarities (potential duplicates):")
for fi in flat_idx:
    i, j = divmod(fi, len(df))
    if (j, i) in seen: continue
    seen.add((i, j))
    print(f"  {all_sims[i,j]:.3f}  {df.iloc[i]['name']:35s} <-> {df.iloc[j]['name']}")

Top product-pair similarities (potential duplicates):
  0.695  Aspen Snow Boots                    <-> Boulder Hiking Boots
  0.609  Iron Combat Boots                   <-> Loam Ankle Boots
  0.604  Cycle Bib Shorts                    <-> Drift Running Shorts


None of these will actually be true duplicates in our catalogue, but you can see how the technique would flag them. In a real product taxonomy, near-duplicates often share a backbone description (manufacturer copy) — their cosine similarity sits above 0.95. Pairs over that threshold get sent to merchandisers to merge or differentiate.